# Paper Visualizations: pyMCAD Style

이 노트북은 논문 작성을 위해 학습 전 데이터(Ground Truth FEA) 및 모델 추론 결과(MGN, FNO 등)를 **pyMCAD_Tuto** 스타일(필드 시각화, 벡터 화살표 Quiver, B-Locus 노드 궤적)로 나란히 비교 출력하고 고해상도로 저장하기 위한 스크립트입니다. 데이터는 컨테이너 환경의 산출물(`.npz`)과 원본 H5 포맷에서 직접 로드됩니다.

In [8]:
import os
import sys
from pathlib import Path
import numpy as np
import h5py
import matplotlib.pyplot as plt
import matplotlib.tri as mtri
from matplotlib.lines import Line2D

# 컨테이너 매핑 경로 및 PyMCAD 라이브러리 추가
BASE_DIR = Path("/workspace/host_data")
if not BASE_DIR.exists():
    # 윈도우 로컬 실행 시 fallback
    BASE_DIR = Path("D:/KDH/NvidiaNemo")

sys.path.append(str(BASE_DIR.parent / "gitEmach" / "eMach"))

print(f"Data Base Directory Setup: {BASE_DIR}")
os.makedirs(BASE_DIR / "paper_figures", exist_ok=True)

Data Base Directory Setup: D:\KDH\NvidiaNemo


## 1. 노드 예측 결과 및 원본 H5 Mesh 데이터 로드

In [9]:
# 모델 추론 결과 로드 (노드 단위)
npz_node_path = BASE_DIR / "field_compare_nodes_allsteps.npz"
d_node = np.load(str(npz_node_path), allow_pickle=True)

node_x_all = d_node["node_x"]
node_y_all = d_node["node_y"]
n_steps = len(node_x_all)

data_dict = {
    "GT": (d_node["gt_bx_node"], d_node["gt_by_node"]),
    "MGN": (d_node["mgn_bx_node"], d_node["mgn_by_node"]),
    "FNO": (d_node["fno_bx_node"], d_node["fno_by_node"]),
    "GINO": (d_node["gino_bx_node"], d_node["gino_by_node"]),
    "RNN": (d_node["rnn_bx_node"], d_node["rnn_by_node"]),
}
times = np.linspace(0, 0.05, n_steps) # 근사 타임스텝

# NPZ 파일에 기록된 실제 case idx를 동적으로 가져옵니다 (기존에 1로 고정되어서 다른 모터의 형상과 맵핑되는 오류 발생)
case_idx = int(d_node.get("case_idx", 5)) # 분석된 case
mesh_triangles = None
element_reg_codes = None # 영역 코드 저장용
h5_file = BASE_DIR / "doe_data" / f"case_{case_idx:04d}" / "postproc" / "Mag_OnLoadTorque_result_1.h5"

if h5_file.exists():
    try:
        with h5py.File(h5_file, 'r') as f:
            n_id = np.asarray(f["mesh/node_id"][:], dtype=np.int32)
            n_1 = np.asarray(f["mesh/node_1"][:], dtype=np.int32)
            n_2 = np.asarray(f["mesh/node_2"][:], dtype=np.int32)
            n_3 = np.asarray(f["mesh/node_3"][:], dtype=np.int32)
            r_c = np.asarray(f["mesh/reg_code"][:], dtype=np.int32) if "mesh/reg_code" in f else None
            
            # LUT for ordering
            max_nid = int(n_id.max()) + 1
            lut = np.full(max_nid, -1, dtype=np.int64)
            for i, nid in enumerate(np.sort(n_id)):
                lut[nid] = i
                
            m = min(len(n_1), len(n_2), len(n_3))
            i1 = lut[np.clip(n_1[:m], 0, max_nid - 1)]
            i2 = lut[np.clip(n_2[:m], 0, max_nid - 1)]
            i3 = lut[np.clip(n_3[:m], 0, max_nid - 1)]
            val = (i1 >= 0) & (i2 >= 0) & (i3 >= 0)
            mesh_triangles = np.column_stack([i1[val], i2[val], i3[val]])
            if r_c is not None:
                element_reg_codes = r_c[val]
                
            print(f"Loaded structured pyMCAD mesh for case_{case_idx:04d}: {len(mesh_triangles)} elements, reg_codes={element_reg_codes is not None}")
    except Exception as e:
        print(f"H5 parsing error: {e}")
else:
    print(f"H5 file not found fallback to pure nodes")

Loaded structured pyMCAD mesh for case_0005: 13288 elements, reg_codes=True


## 2. 정적 플롯 생성 함수 (for Paper Figure Export)

In [10]:
def get_tri(x, y):
    if mesh_triangles is not None:
        # npz의 노드 개수 범위를 벗어나는 인덱스를 참조하는 삼각형(Element) 필터링
        max_idx = len(x)
        valid_mask = (mesh_triangles[:, 0] < max_idx) & \
                     (mesh_triangles[:, 1] < max_idx) & \
                     (mesh_triangles[:, 2] < max_idx)
        valid_tris = mesh_triangles[valid_mask]
        return mtri.Triangulation(x, y, triangles=valid_tris)
    return mtri.Triangulation(x, y)

def plot_paper_fields(step=20, models_to_plot=['GT', 'MGN', 'FNO', 'GINO', 'RNN']):
    """B magnitude field distribution with actual motor mesh structure overlay"""
    fig, axes = plt.subplots(1, len(models_to_plot), figsize=(4.5 * len(models_to_plot), 5), dpi=300, layout='constrained')
    if len(models_to_plot) == 1: axes = [axes]
        
    x, y = node_x_all[step], node_y_all[step]
    tri = get_tri(x, y)
    
    # Calculate global vmin, vmax for color matching
    all_bmag = [np.sqrt(data_dict[m][0][step]**2 + data_dict[m][1][step]**2) for m in models_to_plot]
    vmin = min(b.min() for b in all_bmag)
    vmax = max(b.max() for b in all_bmag)
    
    for ax, name, bmag in zip(axes, models_to_plot, all_bmag):
        m_img = ax.tripcolor(tri, bmag, shading='flat', cmap='jet', vmin=vmin, vmax=vmax)
        ax.triplot(tri, color='k', lw=0.1, alpha=0.15) # structure overlay
        ax.set_title(f"{name} |B| Field", fontsize=16, fontweight='bold')
        ax.set_aspect('equal')
        ax.axis('off') # Clean representation for papers
        
    cbar = fig.colorbar(m_img, ax=axes, orientation='vertical', fraction=0.02, pad=0.02)
    cbar.set_label('|B| [T]', fontsize=14)
    
    plt.savefig(BASE_DIR / f"paper_figures/Mag_Field_Step{step}.png", bbox_inches='tight')
    plt.show()

def plot_paper_quivers(step=20, stride=20, models_to_plot=['GT', 'MGN', 'FNO', 'RNN']):
    """Standard motor CAD style vector fields."""
    fig, axes = plt.subplots(1, len(models_to_plot), figsize=(4.5 * len(models_to_plot), 5), dpi=300, layout='constrained')
    if len(models_to_plot) == 1: axes = [axes]
        
    x, y = node_x_all[step], node_y_all[step]
    tri = get_tri(x, y)
    idx = np.arange(0, x.shape[0], stride)

    for ax, name in zip(axes, models_to_plot):
        bx, by = data_dict[name][0][step], data_dict[name][1][step]
        bmag = np.sqrt(bx[idx]**2 + by[idx]**2)
        
        ax.triplot(tri, color='k', lw=0.1, alpha=0.15)
        m_q = ax.quiver(x[idx], y[idx], bx[idx], by[idx], bmag, cmap='jet', 
                        angles='xy', scale_units='xy', scale=None, width=0.003)
        
        ax.set_title(f"{name} Quiver Vectors", fontsize=16, fontweight='bold')
        ax.set_aspect('equal')
        ax.axis('off')
        
    cbar = fig.colorbar(m_q, ax=axes, orientation='vertical', fraction=0.02, pad=0.02)
    cbar.set_label('|B| [T]', fontsize=14)
    plt.savefig(BASE_DIR / f"paper_figures/Mag_Quiver_Step{step}.png", bbox_inches='tight')
    plt.show()

def plot_paper_locus(models_to_plot=['GT', 'MGN', 'FNO', 'RNN']):
    """B-Locus trace comparison at sampled nodes."""
    fig, axes = plt.subplots(1, len(models_to_plot), figsize=(4.5 * len(models_to_plot), 5), dpi=300, layout='constrained')
    if len(models_to_plot) == 1: axes = [axes]
        
    # Baseline for plotting locus relative offset
    x_base, y_base = node_x_all[0], node_y_all[0]
    tri = get_tri(x_base, y_base)
    stride = 100
    scale = 0.2
    
    sel = np.arange(0, x_base.shape[0], stride)

    for ax, name in zip(axes, models_to_plot):
        bx_all, by_all = data_dict[name]
        
        ax.triplot(tri, color='k', lw=0.1, alpha=0.10)
        ax.scatter(x_base[sel], y_base[sel], s=1, c='k', alpha=0.2)
        
        for ii in sel:
            xc, yc = x_base[ii], y_base[ii]
            # time history for single node ii
            bx_hist = bx_all[:, ii]
            by_hist = by_all[:, ii]
            
            xx = xc + scale * bx_hist
            yy = yc + scale * by_hist
            ax.plot(xx, yy, color='#cc2f2f', linewidth=0.6, alpha=0.6)
            
        ax.set_title(f"{name} B-Locus", fontsize=16, fontweight='bold')
        ax.set_aspect('equal')
        ax.axis('off')
        
    plt.savefig(BASE_DIR / f"paper_figures/Mag_Locus.png", bbox_inches='tight')
    plt.show()

## 3. High-Res Figure 생성 실행 (논문 작성용)

## 4. (추가) Neural Network 출력값을 기존 pyMCAD GUI 플롯에 연동하기
모터 형상(Mesh)이 변경되더라도 구조화되지 않은(Unstructured) 노드 예측 데이터를 기존의 `pyMCAD_Tuto` 노트북에서 사용하던 `interactive_magnetic_plot()` 등의 GUI와 100% 동일하게 그릴 수 있도록 변환 모듈을 적용합니다. 
(학습 대상에 Vector Potential `A`와 Current Density `J` 기능이 추가되었습니다.)

In [ ]:
# 어댑터를 통한 pyMCAD 타임시리즈 객체 변환 및 기존 GUI 플롯 모듈 적용 예시
from nemo_to_pymcad_adapter import convert_nemo_to_pymcad_ts
# pyMCAD_Tuto 노트북 확인 결과에 맞춰 임포트 경로 수정
from tools.motorCAD.pyMCAD import interactive_magnetic_plot, interactive_b_locus_field_plot

n_nodes = len(node_x_all[0])
n_elements = len(mesh_triangles) if mesh_triangles is not None else 0

# 모델 출력 배열: [N_steps, N_nodes, 4] -> (Bx, By, A, J)
ex_pred = np.zeros((n_steps, n_nodes, 4))
for _s in range(n_steps):
    ex_pred[_s, :, 0] = data_dict["GT"][0][_s]  # Bx
    ex_pred[_s, :, 1] = data_dict["GT"][1][_s]  # By
    if "gt_a_node" in d_node and "gt_j_node" in d_node:
        ex_pred[_s, :, 2] = d_node["gt_a_node"][_s]  # 실제 예측값 또는 GT값 A
        ex_pred[_s, :, 3] = d_node["gt_j_node"][_s]  # 실제 예측값 또는 GT값 J
    else:
        ex_pred[_s, :, 2] = np.zeros(n_nodes)
        ex_pred[_s, :, 3] = np.zeros(n_nodes)

if mesh_triangles is not None:
    # 1. pyMCAD의 MagneticRegionsTimeSeries 구조로 변환 (reg_code 추가)
    mock_ts = convert_nemo_to_pymcad_ts(
        node_x_all, 
        node_y_all, 
        mesh_triangles, 
        ex_pred,
        reg_codes=element_reg_codes  # H5에서 추출한 고유 파츠(영역) 코드 전달
    )
    
    # 2. pyMCAD_Tuto 노트북과 100% 동일한 대화형 GUI Plot 실행!
    # step을 이동할때마다 해당 스텝의 상태를 보여줍니다.
    display(interactive_magnetic_plot(mock_ts, quantity="b"))
    # display(interactive_magnetic_plot(mock_ts, quantity="a"))
    # display(interactive_magnetic_plot(mock_ts, quantity="j"))
    
    # --- 주의 ---: Locus Plot(아래)은 '시간별 자속의 궤적(이력) 전체'를 한번에 그리는 기능입니다!
    # 스텝별 변화를 보시려면 위쪽의 interactive_magnetic_plot의 step 슬라이더를 움직이시면 됩니다.
    # display(interactive_b_locus_field_plot(mock_ts))
else:
    print("Mesh Triangles 정보가 없어 pyMCAD 인터페이스로 변환할 수 없습니다.")

None

In [12]:
# 어댑터를 통한 pyMCAD 타임시리즈 객체 변환 및 기존 GUI 플롯 모듈 적용 예시
from nemo_to_pymcad_adapter import convert_nemo_to_pymcad_ts
# pyMCAD_Tuto 노트북 확인 결과에 맞춰 임포트 경로 수정
from tools.motorCAD.pyMCAD import interactive_magnetic_plot, interactive_b_locus_field_plot

n_nodes = len(node_x_all[1])
n_elements = len(mesh_triangles) if mesh_triangles is not None else 0


In [ ]:

# 모델 출력 배열: [N_steps, N_nodes, 4] -> (Bx, By, A, J)
ex_pred = np.zeros((n_steps, n_nodes, 4))
for _s in range(n_steps):
    ex_pred[_s, :, 0] = data_dict["GINO"][0][_s]  # Bx
    ex_pred[_s, :, 1] = data_dict["GINO"][1][_s]  # By
    if "gin_a_node" in d_node and "gin_j_node" in d_node:
        ex_pred[_s, :, 2] = d_node["gin_a_node"][_s]  # 실제 예측값 또는 GT값 A
        ex_pred[_s, :, 3] = d_node["gin_j_node"][_s]  # 실제 예측값 또는 GT값 J
    else:
        ex_pred[_s, :, 2] = np.zeros(n_nodes)
        ex_pred[_s, :, 3] = np.zeros(n_nodes)

if mesh_triangles is not None:
    # 1. pyMCAD의 MagneticRegionsTimeSeries 구조로 변환 (reg_code 추가)
    mock_ts = convert_nemo_to_pymcad_ts(
        node_x_all, 
        node_y_all, 
        mesh_triangles, 
        ex_pred,
        reg_codes=element_reg_codes  # H5에서 추출한 고유 파츠(영역) 코드 전달
    )
    
    # 2. pyMCAD_Tuto 노트북과 100% 동일한 대화형 GUI Plot 실행!
    # step을 이동할때마다 해당 스텝의 상태를 보여줍니다.
    display(interactive_magnetic_plot(mock_ts, quantity="b"))
    # display(interactive_magnetic_plot(mock_ts, quantity="a"))
    # display(interactive_magnetic_plot(mock_ts, quantity="j"))
    
    # --- 주의 ---: Locus Plot(아래)은 '시간별 자속의 궤적(이력) 전체'를 한번에 그리는 기능입니다!
    # 스텝별 변화를 보시려면 위쪽의 interactive_magnetic_plot의 step 슬라이더를 움직이시면 됩니다.
    # display(interactive_b_locus_field_plot(mock_ts))
else:
    print("Mesh Triangles 정보가 없어 pyMCAD 인터페이스로 변환할 수 없습니다.")

None

In [ ]:
import importlib
import sys
import tools.motorCAD.pyMCAD
import tools.motorCAD.pyMCAD.magnetic

# 우리가 패치한 pyMCAD 라이브러리(B: 0~2, A: -0.01~0.01, J: -60~60 제한이 포함된 버전)를 
# 주피터 커널 재시작 없이 바로 반영하기 위해 모듈 리로드 진행
importlib.reload(tools.motorCAD.pyMCAD.magnetic)
importlib.reload(tools.motorCAD.pyMCAD)
from tools.motorCAD.pyMCAD import interactive_magnetic_plot, interactive_b_locus_field_plot

# 모델 출력 배열: [N_steps, N_nodes, 4] -> (Bx, By, A, J)
ex_pred = np.zeros((n_steps, n_nodes, 4))
for _s in range(n_steps):
    # 이 셀은 모델(MGN 혹은 GINO 등) 예측 결과를 담습니다. (현재 코드는 이전 셀과 동일하게 유지)
    if "GINO" in data_dict: # 안전장치
        ex_pred[_s, :, 0] = data_dict["GINO"][0][_s]  # Bx
        ex_pred[_s, :, 1] = data_dict["GINO"][1][_s]  # By
    
    if "gin_a_node" in d_node and "gin_j_node" in d_node:
        ex_pred[_s, :, 2] = d_node["gin_a_node"][_s]  
        ex_pred[_s, :, 3] = d_node["gin_j_node"][_s]  
    else:
        ex_pred[_s, :, 2] = np.zeros(n_nodes)
        ex_pred[_s, :, 3] = np.zeros(n_nodes)

if mesh_triangles is not None:
    # 1. pyMCAD의 MagneticRegionsTimeSeries 구조로 변환 (reg_code 추가)
    mock_ts = convert_nemo_to_pymcad_ts(
        node_x_all,
        node_y_all,
        mesh_triangles,
        ex_pred,
        reg_codes=element_reg_codes  # H5에서 추출한 고유 파츠(영역) 코드 전달  
    )

    # 2. pyMCAD_Tuto 노트북과 100% 동일한 대화형 GUI Plot 실행!
    # step을 이동할때마다 해당 스텝의 상태를 보여줍니다.
    print("=== GINO 모델 시각화 (새로 적용된 Colorbar Limits 반영) ===")
    display(interactive_magnetic_plot(mock_ts, quantity="b"))
    
    # --- 주의 ---: Locus Plot(아래)은 '시간별 자속의 궤적(이력) 전체'를 한번에 그리는 기능입니다!
    # 스텝별 변화를 보시려면 위쪽의 interactive_magnetic_plot의 step 슬라이더를 움직이시면 됩니다.
    # display(interactive_b_locus_field_plot(mock_ts))
else:
    print("Mesh Triangles 정보가 없어 pyMCAD 인터페이스로 변환할 수 없습니다.")

=== GINO 모델 시각화 (새로 적용된 Colorbar Limits 반영) ===


None

## 5. 모델간 예측 차이(Error) 시각화 인터랙티브 GUI
두 개의 모델 예측값 또는 Ground Truth(GT)와의 오차(Difference)를 구해서 시각화하는 모듈입니다.
(B, A, J 각각의 오차 스케일에 맞춰 Colorbar의 `vmin`, `vmax`가 0을 중심으로 고정됩니다.)

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt

# 1. 모델 간 Error 값을 구하기 (예: GINO 예측값 - GT 실제값)
# Percentage Error와 Absolute Error 2가지를 모두 만듭니다.
diff_pred_pct = np.zeros((n_steps, n_nodes, 4))
diff_pred_abs = np.zeros((n_steps, n_nodes, 4))

for _s in range(n_steps):
    gt_b_mag = np.sqrt(data_dict["GT"][0][_s]**2 + data_dict["GT"][1][_s]**2)
    mgn_b_mag = np.sqrt(data_dict["GINO"][0][_s]**2 + data_dict["GINO"][1][_s]**2)
    
    # [Percent Error] B의 경우 오차 % 스칼라값을 X에 넣고 Y는 0으로 처리 (magnitude가 %오차가 됨)
    diff_pred_pct[_s, :, 0] = ((mgn_b_mag - gt_b_mag) / (gt_b_mag + 1e-4)) * 100
    diff_pred_pct[_s, :, 1] = 0.0
    
    # [Absolute Error] B의 크기 차이 절대값을 X에 넣고 Y는 0으로 처리. (0 ~ 0.2T)
    # pyMCAD는 B를 magnitude로 그리므로 X에 양수 절대오차를 넣어줍니다. 
    diff_pred_abs[_s, :, 0] = np.abs(mgn_b_mag - gt_b_mag)
    diff_pred_abs[_s, :, 1] = 0.0

    if "gt_a_node" in d_node and "gin_a_node" in d_node:
        gt_a = d_node["gt_a_node"][_s]
        gt_j = d_node["gt_j_node"][_s]
        mgn_a = d_node["gin_a_node"][_s]
        mgn_j = d_node["gin_j_node"][_s]
        
        # Percentage
        diff_pred_pct[_s, :, 2] = ((mgn_a - gt_a) / (np.abs(gt_a) + 1e-6)) * 100
        diff_pred_pct[_s, :, 3] = ((mgn_j - gt_j) / (np.abs(gt_j) + 1e-6)) * 100
        
        # Absolute 
        diff_pred_abs[_s, :, 2] = mgn_a - gt_a
        diff_pred_abs[_s, :, 3] = mgn_j - gt_j

if mesh_triangles is not None:
    # 2. 오차 데이터를 pyMCAD 타임시리즈 객체로 2개 세트로 변환
    diff_ts_pct = convert_nemo_to_pymcad_ts(
        node_x_all, node_y_all, mesh_triangles, diff_pred_pct, reg_codes=element_reg_codes
    )
    diff_ts_abs = convert_nemo_to_pymcad_ts(
        node_x_all, node_y_all, mesh_triangles, diff_pred_abs, reg_codes=element_reg_codes
    )

    # 파트(reg_code) 목록 추출
    unique_parts = ["All"]
    if element_reg_codes is not None:
        unique_parts += [str(p) for p in sorted(list(np.unique(element_reg_codes)))]

    # 3. 오차 전용 인터랙티브 GUI 정의
    def interactive_difference_plot(ts_pct, ts_abs, initial_step=None):
        if len(ts_pct) == 0: return display("Empty time series")
        steps = ts_pct.steps
        if initial_step is None: initial_step = steps[0]

        type_dd = widgets.Dropdown(options=["Absolute", "Percentage (%)"], value="Absolute", description="Type:")
        step_slider = widgets.SelectionSlider(options=steps, value=initial_step, description="step", continuous_update=False, layout=widgets.Layout(width="400px"))
        qty_dd = widgets.Dropdown(options=[("B Error", "b"), ("A Error", "a"), ("J Error", "j")], value="b", description="qty:")
        part_dd = widgets.Dropdown(options=unique_parts, value="All", description="Part:")
        mesh_chk = widgets.Checkbox(value=False, description="mesh", indent=False)
        out = widgets.Output()

        # 오차 플롯을 위한 Colorbar 제한값 세팅
        diff_limits = {
            "Percentage (%)": {
                "b": {"vmin": 0.0, "vmax": 50.0},       # 0% to 50% Error
                "a": {"vmin": -100.0, "vmax": 100.0},   # -100% to 100% Error
                "j": {"vmin": -100.0, "vmax": 100.0}    # -100% to 100% Error
            },
            "Absolute": {
                "b": {"vmin": 0.0, "vmax": 0.2},       # B 오차 (절대값 크기, 0 ~ 0.2T)
                "a": {"vmin": -0.002, "vmax": 0.002},  # A 절대오차
                "j": {"vmin": -15.0, "vmax": 15.0}     # J 절대오차
            }
        }

        _last_fig = {"fig": None}

        def _draw(*_):
            with out:
                out.clear_output(wait=True)
                if _last_fig["fig"] is not None:
                    plt.close(_last_fig["fig"])

                fig, ax = plt.subplots(layout="constrained")
                _last_fig["fig"] = fig

                err_type = type_dd.value
                step = int(step_slider.value)
                qty = str(qty_dd.value).lower()
                part_val = None if part_dd.value == "All" else int(part_dd.value)
                
                # 사용할 데이터 소스와 리밋
                active_ts = ts_abs if err_type == "Absolute" else ts_pct
                lim = diff_limits[err_type].get(qty, {"vmin": 0.0, "vmax": 0.2})
                
                # B는 pyMCAD의 magnitude 연산 때문에 양수만 존재. (Reds 컬러맵 권장, Absolute의 경우 Jet도 가능하지만 오차는 Reds 선호)
                if qty == "b":
                    cmap = "jet" if err_type == "Absolute" else "Reds"
                else: # A, J는 양/음수가 모두 있으므로 발산형(bwr)
                    cmap = "bwr"

                # eMach(pyMCAD)의 plot 메서드 호출 (패치된 vmin/vmax, reg_code 활용)
                ax = active_ts.by_step[step].plot(
                    reg_code=part_val,
                    quantity=qty,
                    s=4 if part_val is not None else 2, # 특정 파트 필터링시에 점 크기 확대로 가시성 증가
                    cmap=cmap,
                    vmin=lim["vmin"],
                    vmax=lim["vmax"],
                    ax=ax,
                    show=False,
                    mesh=bool(mesh_chk.value)
                )
                
                unit = "(%)" if err_type == "Percentage (%)" else "(Absolute)"
                ax.set_title(f"GINO vs GT Error {unit} - {qty.upper()} [Step: {step}]")
                plt.show()

        type_dd.observe(_draw, names="value")
        step_slider.observe(_draw, names="value")
        qty_dd.observe(_draw, names="value")
        part_dd.observe(_draw, names="value")
        mesh_chk.observe(_draw, names="value")

        control_box = widgets.VBox([
            widgets.HBox([type_dd, qty_dd, part_dd]),
            widgets.HBox([step_slider, mesh_chk])
        ])
        display(widgets.VBox([control_box, out]))
        _draw()

    print("=== GINO 모델과 GT의 스텝별 Error 분석 (Percent & Absolute, 파트별 보기 지원) ===")
    interactive_difference_plot(diff_ts_pct, diff_ts_abs)
else:
    print("메시 정보가 없어 Error Plot을 생성할 수 없습니다.")

=== GINO 모델과 GT의 스텝별 Percentage Error (%) 분석 ===
